# 전체 이력 GPU 비교: LightGBM · CatBoost

예측 기준일 `t`까지의 판매 이력으로 `t+1`~`t+7`을 예측합니다. 동일한 전체 이력 Feature와 시간순 검증 구간을 두 모델에 사용합니다. 마지막 28개 기준일은 검증으로 분리하고, 그 기간 이후 정답은 학습에 넣지 않습니다.

**실행 전:** Colab 런타임을 NVIDIA GPU 및 가능한 경우 높은 RAM으로 설정하세요. 아래 셀은 My Drive에 저장소를 clone하지만, Git에 없는 Kaggle 원본 CSV는 `data/raw`에 별도로 넣어야 합니다. 첫 Feature 생성과 두 번의 전체 학습은 오래 걸리므로 각 셀을 의도적으로 실행하세요.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/hyun0415/retail-demand-forecasting.git"
REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
if not REPO.parent.is_dir():
    raise FileNotFoundError(f"My Drive 경로를 찾지 못했습니다: {REPO.parent}")
if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(
        f"저장소 경로에 폴더가 있지만 .git이 없습니다: {REPO}. "
        "중단된 clone의 잔여물인지 먼저 확인한 뒤 이름을 바꾸거나 정리하세요."
    )
if not REPO.exists():
    cloned = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        capture_output=True, text=True,
    )
    if cloned.returncode:
        raise RuntimeError(
            f"git clone 실패 (exit {cloned.returncode}):\n{cloned.stderr.strip()}"
        )
if not (REPO / ".git").is_dir():
    raise RuntimeError(f"저장소 경로가 아닙니다: {REPO}")
os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 1. 환경과 GPU 확인

CatBoost는 설치 패키지의 GPU 옵션을 사용합니다. LightGBM은 일반 설치만으로 CUDA 지원이 보장되지 않으므로 공식 빌드 옵션으로 다시 설치합니다. CUDA Toolkit의 `nvcc`가 없는 런타임에서는 아래 단계가 중단됩니다.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
required = (REPO / "requirements.txt").read_text(encoding="utf-8").splitlines()
packages = [line for line in required if line.strip() and not line.startswith("lightgbm")]
subprocess.run([sys.executable, "-m", "pip", "install", *packages, "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "lightgbm"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-binary=lightgbm", "--no-deps",
    "--config-settings=cmake.define.USE_CUDA=ON",
    "--config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES=native",
    "lightgbm==4.7.0",
], check=True)

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import pandas as pd

toy_x = pd.DataFrame(np.arange(240, dtype=np.float32).reshape(80, 3))
toy_x["family"] = pd.Categorical(["A", "B"] * 40)
toy_y = np.arange(80, dtype=np.float32)
LGBMRegressor(
    n_estimators=2, min_child_samples=2, objective="regression_l1",
    device_type="cuda", verbosity=-1,
).fit(toy_x, toy_y)
toy_x["family"] = toy_x["family"].astype(str)
CatBoostRegressor(
    iterations=2, loss_function="MAE", task_type="GPU", devices="0", verbose=False,
).fit(toy_x, toy_y, cat_features=["family"])
print("두 모델의 GPU 시험 학습이 완료됐습니다.")

## 2. Kaggle 원본과 전체 이력 Feature

`train.csv`, `stores.csv`, `holidays_events.csv`, `transactions.csv`를 clone한 저장소의 `data/raw`에 넣으세요. 원본과 생성 파일은 Git 추적 대상이 아닙니다. Feature는 28일 단위로 나누어 Parquet에 저장하며, 이미 완성된 캐시는 다시 만들지 않습니다.

In [ ]:
required_files = ("train.csv", "stores.csv", "holidays_events.csv", "transactions.csv")
missing = [name for name in required_files if not (REPO / "data/raw" / name).is_file()]
if missing:
    raise FileNotFoundError(f"data/raw에 다음 파일을 넣으세요: {', '.join(missing)}")
subprocess.run([
    sys.executable, "scripts/build_full_features.py",
    "--config", "configs/full_lightgbm_gpu.toml",
    "--chunk-days", "28",
], cwd=REPO, check=True)

## 3. 모델별 전체 이력 학습

두 설정은 같은 Feature 캐시와 검증 구간을 사용합니다. LightGBM 실행이 끝난 뒤 CatBoost를 실행하세요. 각 실행은 `models/full_history`, `outputs/full_history`에 저장되어 기존 365일 실험 결과를 덮어쓰지 않습니다.

In [ ]:
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from retail_forecast.config import load_config
from retail_forecast.pipeline import feature_cache_path, run_experiment

lightgbm_config = load_config(REPO / "configs/full_lightgbm_gpu.toml")
if not feature_cache_path(lightgbm_config).exists():
    raise FileNotFoundError("전체 이력 Feature 캐시가 없습니다. 이전 셀을 먼저 실행하세요.")
lightgbm_result = run_experiment(lightgbm_config, progress=True)
print(lightgbm_result)

In [ ]:
import gc

gc.collect()
catboost_config = load_config(REPO / "configs/full_catboost_gpu.toml")
if feature_cache_path(catboost_config) != feature_cache_path(lightgbm_config):
    raise ValueError("두 모델의 Feature 캐시 경로가 다릅니다.")
catboost_result = run_experiment(catboost_config, progress=True)
print(catboost_result)

## 4. 결과 표 저장

전체 일별 예측, horizon별 예측, 발주용 7일 합계의 오차를 별도 표로 저장합니다. 두 모델의 결과 파일이 모두 있을 때만 비교표를 만듭니다.

In [ ]:
import json
import pandas as pd

from retail_forecast.metrics import regression_metrics

result_dir = REPO / "outputs/full_history"
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
rows = []
horizon_tables = []
for model_name in ("lightgbm", "catboost"):
    metrics_path = result_dir / f"metrics_{model_name}.json"
    summary_path = result_dir / f"decision_summary_{model_name}.csv"
    horizon_path = result_dir / f"metrics_by_horizon_{model_name}.csv"
    for path in (metrics_path, summary_path, horizon_path):
        if not path.is_file():
            raise FileNotFoundError(f"모델 실행 결과가 없습니다: {path}")
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary = pd.read_csv(summary_path, usecols=["actual_sum_1_7", "prediction_sum_1_7"])
    if summary.isna().any().any():
        raise ValueError(f"{model_name}: 7일 합계에 결측값이 있습니다")
    weekly = regression_metrics(
        summary["actual_sum_1_7"].to_numpy(),
        summary["prediction_sum_1_7"].to_numpy(),
    )
    rows.append({
        "model": model_name, "training_scope": "all_available_origins",
        "git_commit": commit, "validation_origin_start": metrics["validation_origin_start"],
        "train_rows": metrics["train_rows"], "validation_rows": metrics["validation_rows"],
        **{name: metrics[name] for name in ("rmsle", "mae", "wape", "bias")},
        **{f"sum_1_7_{name}": value for name, value in weekly.items()},
    })
    horizon = pd.read_csv(horizon_path)
    horizon.insert(0, "model", model_name)
    horizon_tables.append(horizon)
comparison = pd.DataFrame(rows)
by_horizon = pd.concat(horizon_tables, ignore_index=True)
comparison.to_csv(result_dir / "comparison_full_history.csv", index=False)
by_horizon.to_csv(result_dir / "comparison_by_horizon_full_history.csv", index=False)
display(comparison)
display(by_horizon)
print("저장 위치:", result_dir)

### 해석과 주의사항

- 전체 이력은 *모든 기준일*을 사용한다는 뜻입니다. 마지막 28개 기준일의 목표 판매량은 학습에서 제외합니다.
- 이전 365일 결과와 학습 기간과 Feature 조건이 달라 숫자를 직접 비교하면 안 됩니다. 이번 설정은 예측 시점에 알 수 없는 지진·돌발 이벤트 변수를 제외합니다.
- 목표일 프로모션은 **7일 앞까지 계획값을 실제로 확보할 수 있을 때만** 유효합니다. 계획값이 없다면 두 설정에서 `target_onpromotion`을 제외하고 Feature 캐시를 그대로 사용해 재학습해야 합니다.
- `sales`와 `current_transactions`는 기준일 영업 종료 후 예측한다는 전제입니다. 영업 중 발주라면 이 변수도 제외해야 합니다.
- 전체 이력 표는 메모리를 많이 사용합니다. Colab의 GPU와 RAM 할당은 변할 수 있고, My Drive I/O는 느릴 수 있습니다. 런타임이 끊겨도 완성된 Parquet과 결과 파일은 Drive에 남습니다. `.building` 폴더가 남았다면 생성이 중단된 것이므로 내용 확인 후 수동으로 정리하세요.